In [1]:
import requests
import xml.etree.ElementTree as ET
import os
import pymysql
import time
import itertools
from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()
service_key = os.getenv('PUBLIC_SERVICE_KEY')

# DB 연결
db = pymysql.connect(
    host='localhost',
    user='root',          
    password='qwer123456@@',  # 선생님의 비밀번호
    db='carmaster_db',    
    charset='utf8mb4'
)
cursor = db.cursor()

# ==========================================
# 1. 수집 조건 세팅
# ==========================================
vhcty_codes  = ['1', '2', '3', '4'] 
region_codes = [str(i) for i in range(1, 18)] 
fuel_codes   = ['2', '5', '7', '8'] 
import_codes = ['국산', '외산'] 

valid_sex_age = []
for age in range(1, 9):
    valid_sex_age.append(('남자', str(age)))
    valid_sex_age.append(('여자', str(age)))

all_combinations = list(itertools.product(
    vhcty_codes, region_codes, fuel_codes, import_codes, valid_sex_age
))

url = 'https://apis.data.go.kr/B553881/newRegistlnfoService_02/getnewRegistlnfoService02'

insert_sql = """
    INSERT INTO car_registration_stats 
    (regist_yy, regist_mt, vhcty_asort_code, regist_grc_code, use_fuel_code, 
     prpos_se_code, sexdstn, agrde, hmmd_imp_se_nm, cnt)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

print(f"총 {len(all_combinations)}개의 조건으로 2025년 10월 데이터를 수집합니다.")

# ==========================================
# 2. 2025년 10월 전용 수집 루프
# ==========================================
for year in [2025]:
    print(f"\n--- {year}년 10월 자가용 데이터 수집 시작 ---")
    
    for combo in all_combinations:
        vhcty, region, fuel, imp, (sex, age) = combo
        
        params = {
            'serviceKey'     : service_key,
            'registYy'       : str(year),
            'registMt'       : '10',         # 🌟 10월로 고정!
            'vhctyAsortCode' : vhcty,
            'registGrcCode'  : region,
            'useFuelCode'    : fuel,
            'prposSeNm'      : '1',          
            'sexdstn'        : sex,
            'agrde'          : age,
            'hmmdImpSeNm'    : imp
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            root = ET.fromstring(response.text)
            
            count = root.findtext('.//dtaCo')
            
            if count and count.strip().isdigit() and int(count) > 0:
                # 🌟 DB에도 '10'월로 정확하게 저장되도록 설정!
                cursor.execute(insert_sql, (
                    str(year), '10', vhcty, region, fuel, '1', sex, age, imp, int(count)
                ))
                db.commit()
                print(f"✅ [{year}-10] 차종:{vhcty}, 지역:{region}, 연료:{fuel}, {sex}({age}0대), {imp} -> {count}건")
                
            time.sleep(0.1) # 서버 차단 방지용 
            
        except ET.ParseError:
            print(f"🚨 일시적 서버 오류 또는 트래픽 한도 도달 (파라미터: {params})")
        except Exception as e:
            print(f"⚠️ 기타 에러 발생: {e}")

db.close()
print("\n🎉 2025년 10월 데이터 수집이 완벽하게 끝났습니다!")

c:\Users\Playdata\miniconda3\envs\class\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


총 8704개의 조건으로 2025년 10월 데이터를 수집합니다.

--- 2025년 10월 자가용 데이터 수집 시작 ---
✅ [2025-10] 차종:1, 지역:1, 연료:2, 남자(20대), 국산 -> 1건
✅ [2025-10] 차종:1, 지역:1, 연료:2, 남자(30대), 국산 -> 4건
✅ [2025-10] 차종:1, 지역:1, 연료:2, 남자(40대), 국산 -> 3건
✅ [2025-10] 차종:1, 지역:1, 연료:2, 여자(40대), 국산 -> 1건
✅ [2025-10] 차종:1, 지역:1, 연료:2, 남자(50대), 국산 -> 13건
✅ [2025-10] 차종:1, 지역:1, 연료:2, 여자(50대), 국산 -> 5건
✅ [2025-10] 차종:1, 지역:1, 연료:2, 남자(60대), 국산 -> 3건
✅ [2025-10] 차종:1, 지역:1, 연료:2, 남자(70대), 국산 -> 1건
✅ [2025-10] 차종:1, 지역:1, 연료:2, 여자(70대), 국산 -> 1건
✅ [2025-10] 차종:1, 지역:1, 연료:2, 남자(50대), 외산 -> 1건
✅ [2025-10] 차종:1, 지역:1, 연료:5, 남자(20대), 국산 -> 18건
✅ [2025-10] 차종:1, 지역:1, 연료:5, 여자(20대), 국산 -> 13건
✅ [2025-10] 차종:1, 지역:1, 연료:5, 남자(30대), 국산 -> 90건
✅ [2025-10] 차종:1, 지역:1, 연료:5, 여자(30대), 국산 -> 50건
✅ [2025-10] 차종:1, 지역:1, 연료:5, 남자(40대), 국산 -> 167건
✅ [2025-10] 차종:1, 지역:1, 연료:5, 여자(40대), 국산 -> 51건
✅ [2025-10] 차종:1, 지역:1, 연료:5, 남자(50대), 국산 -> 126건
✅ [2025-10] 차종:1, 지역:1, 연료:5, 여자(50대), 국산 -> 40건
✅ [2025-10] 차종:1, 지역:1, 연료:5, 남자(60대), 국산 -> 47건
✅ [2025

In [5]:
%pip install pymysql
%pip install cryptography

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ------------------------ --------------- 2.1/3.5 MB 10.6 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 10.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install xmltodict

Note: you may need to restart the kernel to use updated packages.


In [26]:
import xmltodict
data = xmltodict.parse(response.text)
print(data)
data.get('response').get('body').get('dtaCo')

{'response': {'header': {'resultCode': '00', 'resultMsg': 'NORMAL_CODE'}, 'body': {'dtaCo': '9872'}}}


'9872'